# Investment Project GUI Launcher

이 노트북은 터미널 명령어 대신 **버튼과 선택 메뉴**로 프로젝트 기능을 실행하기 위한 화면입니다.

## 사용 순서

1. 이 파일을 `OneDrive\주식\investment\RUN_PROJECT.ipynb`에 둡니다.
2. Jupyter Notebook 또는 JupyterLab에서 엽니다.
3. 아래의 **1회 설치 셀**을 먼저 실행합니다.
4. 그다음 GUI 셀을 실행하고 원하는 버튼만 누릅니다.

각 기능은 따로 실행되며, 전체 작업이 자동으로 한꺼번에 실행되지는 않습니다.


In [ ]:
# 1회 설치 셀: 처음 사용할 때 또는 requirements가 바뀐 뒤에만 실행
from pathlib import Path
import subprocess
import sys

REPO = Path.cwd().resolve()

if not (REPO / "pyproject.toml").exists():
    raise RuntimeError(
        "이 노트북을 investment 저장소 바로 아래에 두고 실행하세요.\n"
        f"현재 위치: {REPO}"
    )

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", ".", "ipywidgets"]
)

print("설치 완료")
print("프로젝트 위치:", REPO)


## GUI 실행

아래 셀을 실행하면 가격 업데이트, 전처리, 최적화, 시뮬레이션, 재무 분석, Transformer 학습 버튼이 나타납니다.


In [ ]:

from __future__ import annotations

from datetime import date
from pathlib import Path
import subprocess
import sys
import traceback

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from stock_research.indicators import add_indicators, normalize_price_columns
from stock_research.io_utils import atomic_to_csv, read_csv_fallback
from stock_research.paths import load_paths
from stock_research.tickers import load_tickers

REPO = Path.cwd().resolve()
paths = load_paths()
ticker_configs = load_tickers(paths.repo_root / "config" / "tickers.json")
ticker_options = sorted(ticker_configs)
default_ticker = "TSLA" if "TSLA" in ticker_configs else ticker_options[0]
today_text = date.today().isoformat()


def stream_script(script_name: str, arguments: list[str], output: widgets.Output) -> int:
    output.clear_output()
    command = [sys.executable, str(REPO / "scripts" / script_name), *arguments]

    with output:
        print("실행 명령:")
        print(" ".join(f'"{item}"' if " " in item else item for item in command))
        print("-" * 80)

        process = subprocess.Popen(
            command,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")

        return_code = process.wait()
        print("-" * 80)

        if return_code == 0:
            print("완료")
        else:
            print(f"실패: 종료 코드 {return_code}")

    return return_code


def preprocess_one_ticker(ticker: str) -> Path:
    config = ticker_configs[ticker]
    company_dir = paths.raw_prices / config.display_name

    if not company_dir.exists():
        raise FileNotFoundError(
            f"가격 폴더가 없습니다: {company_dir}\\n"
            "먼저 가격 업데이트 버튼을 실행하세요."
        )

    frames: list[pd.DataFrame] = []

    for csv_path in sorted(company_dir.glob("*.csv")):
        try:
            frame = normalize_price_columns(read_csv_fallback(csv_path))
            frames.append(frame)
        except Exception as exc:
            print(f"SKIP {csv_path.name}: {exc}")

    if not frames:
        raise RuntimeError(f"처리 가능한 CSV가 없습니다: {company_dir}")

    combined = (
        pd.concat(frames, ignore_index=True)
        .sort_values("날짜")
        .drop_duplicates("날짜", keep="last")
        .reset_index(drop=True)
    )

    processed = add_indicators(combined, drop_warmup=False)
    output = paths.processed / f"{config.display_name}_지표포함.csv"
    atomic_to_csv(processed, output, index=False)
    return output


# -------------------------------------------------------------------
# 공통 경로 표시
# -------------------------------------------------------------------
path_output = widgets.Output()

with path_output:
    print("Repository :", paths.repo_root)
    print("Stock root :", paths.stock_root)
    print("Raw prices:", paths.raw_prices)
    print("Processed :", paths.processed)
    print("Macro     :", paths.macro)
    print("Results   :", paths.results)


# -------------------------------------------------------------------
# 가격 업데이트 및 전처리
# -------------------------------------------------------------------
price_tickers = widgets.SelectMultiple(
    options=ticker_options,
    value=(default_ticker,),
    description="Tickers",
    rows=8,
    layout=widgets.Layout(width="360px"),
)

update_price_button = widgets.Button(
    description="선택 종목 가격 업데이트",
    button_style="primary",
)

preprocess_button = widgets.Button(
    description="선택 종목 전처리",
)

update_macro_button = widgets.Button(
    description="Macro 업데이트",
)

price_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #999", max_height="380px", overflow="auto")
)


def on_update_prices(_):
    selected = list(price_tickers.value)
    if not selected:
        with price_output:
            price_output.clear_output()
            print("종목을 하나 이상 선택하세요.")
        return

    arguments: list[str] = []
    for ticker in selected:
        arguments.extend(["--ticker", ticker])

    stream_script("update_prices.py", arguments, price_output)


def on_preprocess(_):
    price_output.clear_output()

    with price_output:
        for ticker in price_tickers.value:
            try:
                output = preprocess_one_ticker(ticker)
                print(f"{ticker}: {output}")
            except Exception:
                print(f"{ticker}: 실패")
                traceback.print_exc()


def on_update_macro(_):
    stream_script("update_macro.py", [], price_output)


update_price_button.on_click(on_update_prices)
preprocess_button.on_click(on_preprocess)
update_macro_button.on_click(on_update_macro)

price_panel = widgets.VBox(
    [
        widgets.HTML("<h3>1. 가격 데이터와 전처리</h3>"),
        price_tickers,
        widgets.HBox([update_price_button, preprocess_button, update_macro_button]),
        price_output,
    ]
)


# -------------------------------------------------------------------
# 최적화
# -------------------------------------------------------------------
opt_strategy = widgets.Dropdown(
    options=[("VIX 전략", "vix"), ("Technical 전략", "technical")],
    value="vix",
    description="전략",
)

opt_ticker = widgets.Dropdown(
    options=ticker_options,
    value=default_ticker,
    description="Ticker",
)

opt_start = widgets.Text(value="2022-01-01", description="Start")
opt_end = widgets.Text(value=today_text, description="End")
opt_tpe = widgets.BoundedIntText(value=20, min=1, max=100000, description="TPE trials")
opt_cma = widgets.BoundedIntText(value=10, min=0, max=100000, description="CMA trials")

opt_button = widgets.Button(
    description="선택 전략 최적화 실행",
    button_style="warning",
)

opt_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #999", max_height="380px", overflow="auto")
)


def on_optimize(_):
    ticker = opt_ticker.value
    company = ticker_configs[ticker].display_name

    script = (
        "optimize_vix.py"
        if opt_strategy.value == "vix"
        else "optimize_technical.py"
    )

    arguments = [
        company,
        "--start",
        opt_start.value,
        "--end",
        opt_end.value,
        "--tpe-trials",
        str(opt_tpe.value),
        "--cma-trials",
        str(opt_cma.value),
    ]

    stream_script(script, arguments, opt_output)


opt_button.on_click(on_optimize)

optimization_panel = widgets.VBox(
    [
        widgets.HTML("<h3>2. 전략 최적화</h3>"),
        widgets.HBox([opt_strategy, opt_ticker]),
        widgets.HBox([opt_start, opt_end]),
        widgets.HBox([opt_tpe, opt_cma]),
        opt_button,
        opt_output,
    ]
)


# -------------------------------------------------------------------
# 시뮬레이션
# -------------------------------------------------------------------
sim_strategy = widgets.Dropdown(
    options=[("VIX 전략", "vix"), ("Technical 전략", "technical")],
    value="vix",
    description="전략",
)

sim_index = widgets.BoundedIntText(value=1, min=1, max=1000000, description="Index")
sim_start = widgets.Text(value="2022-01-01", description="Start")
sim_end = widgets.Text(value=today_text, description="End")
sim_extra = widgets.Checkbox(value=False, description="매수 신호 때 추가 투자")
sim_dca = widgets.FloatText(value=10000.0, description="Daily DCA")

sim_button = widgets.Button(
    description="시뮬레이션 실행",
    button_style="success",
)

sim_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #999", max_height="380px", overflow="auto")
)


def on_simulate(_):
    script = (
        "simulate_vix.py"
        if sim_strategy.value == "vix"
        else "simulate_technical.py"
    )

    arguments = [
        str(sim_index.value),
        "--start",
        sim_start.value,
        "--end",
        sim_end.value,
        "--daily-dca-amount",
        str(sim_dca.value),
    ]

    if sim_extra.value:
        arguments.append("--extra-on-buy")

    stream_script(script, arguments, sim_output)


sim_button.on_click(on_simulate)

simulation_panel = widgets.VBox(
    [
        widgets.HTML("<h3>3. 시뮬레이션</h3>"),
        widgets.HBox([sim_strategy, sim_index]),
        widgets.HBox([sim_start, sim_end]),
        widgets.HBox([sim_extra, sim_dca]),
        sim_button,
        sim_output,
    ]
)


# -------------------------------------------------------------------
# 재무 데이터
# -------------------------------------------------------------------
fin_ticker = widgets.Dropdown(
    options=ticker_options,
    value=default_ticker,
    description="Ticker",
)

fin_headless = widgets.Checkbox(value=True, description="브라우저 숨김")
fin_wait = widgets.FloatText(value=5.0, description="Wait sec")

scrape_q_button = widgets.Button(description="분기 재무 크롤링")
scrape_a_button = widgets.Button(description="연간 재무 크롤링")
analyze_button = widgets.Button(description="재무 분석")
visualize_button = widgets.Button(description="재무 그래프")

fin_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #999", max_height="380px", overflow="auto")
)


def run_scrape(frequency: str):
    arguments = [
        "--ticker",
        fin_ticker.value,
        "--frequency",
        frequency,
        "--wait",
        str(fin_wait.value),
    ]

    if fin_headless.value:
        arguments.append("--headless")

    stream_script("scrape_financials.py", arguments, fin_output)


def on_scrape_q(_):
    run_scrape("Q")


def on_scrape_a(_):
    run_scrape("A")


def on_analyze(_):
    stream_script(
        "analyze_financials.py",
        ["--ticker", fin_ticker.value],
        fin_output,
    )


def on_visualize(_):
    stream_script(
        "visualize_financials.py",
        [fin_ticker.value],
        fin_output,
    )


scrape_q_button.on_click(on_scrape_q)
scrape_a_button.on_click(on_scrape_a)
analyze_button.on_click(on_analyze)
visualize_button.on_click(on_visualize)

financial_panel = widgets.VBox(
    [
        widgets.HTML("<h3>4. 재무 데이터</h3>"),
        widgets.HBox([fin_ticker, fin_headless, fin_wait]),
        widgets.HBox(
            [scrape_q_button, scrape_a_button, analyze_button, visualize_button]
        ),
        fin_output,
    ]
)


# -------------------------------------------------------------------
# Transformer
# -------------------------------------------------------------------
ml_ticker = widgets.Dropdown(
    options=ticker_options,
    value=default_ticker,
    description="Ticker",
)

ml_start = widgets.Text(value="2022-01-01", description="Start")
ml_end = widgets.Text(value=today_text, description="End")
ml_epochs = widgets.BoundedIntText(value=5, min=1, max=10000, description="Epochs")
ml_horizon = widgets.BoundedIntText(value=20, min=1, max=1000, description="Horizon")

ml_button = widgets.Button(
    description="Transformer 학습",
    button_style="danger",
)

ml_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #999", max_height="380px", overflow="auto")
)


def on_train_transformer(_):
    ticker = ml_ticker.value
    company = ticker_configs[ticker].display_name

    arguments = [
        company,
        "--start",
        ml_start.value,
        "--end",
        ml_end.value,
        "--epochs",
        str(ml_epochs.value),
        "--horizon",
        str(ml_horizon.value),
    ]

    stream_script("train_transformer.py", arguments, ml_output)


ml_button.on_click(on_train_transformer)

transformer_panel = widgets.VBox(
    [
        widgets.HTML("<h3>5. Transformer</h3>"),
        widgets.HBox([ml_ticker, ml_epochs, ml_horizon]),
        widgets.HBox([ml_start, ml_end]),
        ml_button,
        ml_output,
    ]
)


# -------------------------------------------------------------------
# 테스트
# -------------------------------------------------------------------
test_button = widgets.Button(description="프로젝트 테스트 실행")
test_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #999", max_height="300px", overflow="auto")
)


def on_test(_):
    test_output.clear_output()
    command = [sys.executable, "-m", "pytest"]

    with test_output:
        process = subprocess.Popen(
            command,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")

        process.wait()


test_button.on_click(on_test)

test_panel = widgets.VBox(
    [
        widgets.HTML("<h3>6. 테스트</h3>"),
        test_button,
        test_output,
    ]
)


tabs = widgets.Tab(
    children=[
        price_panel,
        optimization_panel,
        simulation_panel,
        financial_panel,
        transformer_panel,
        test_panel,
    ]
)

for index, title in enumerate(
    ["가격/전처리", "최적화", "시뮬레이션", "재무", "Transformer", "테스트"]
):
    tabs.set_title(index, title)

display(path_output)
display(tabs)


## GitHub에 변경사항 저장하기

이 노트북이나 Python 파일을 수정한 뒤에는 GitHub Desktop에서:

1. 변경 파일 확인
2. Summary 작성
3. **Commit to main**
4. **Push origin**

실제 가격 데이터, 재무 데이터, 결과 파일은 OneDrive에만 저장되고 GitHub에는 올라가지 않도록 구성되어 있습니다.
